# 📈 Module 1c: R Graphics & Statistical Analysis for Genomics
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Course:** Bioinformatics: From Bulk & scRNA-seq to Spatial Transcriptomics  
**Focus:** Data Reshaping (`reshape2`), Base R plots, `ggplot2` Publication Graphics, Volcano Plots, Hypothesis Testing (t-test, Wilcoxon), FDR corrections, and PCA.

---
### 🎯 Learning Objectives:
1. Reshape data from Wide to Long format using `reshape2::melt()`.
2. Generate publication-ready Volcano plots, Boxplots, and Heatmaps with `ggplot2`.
3. Perform hypothesis testing (t-test, Wilcoxon rank-sum) and calculate Adjusted P-values (FDR / Benjamini-Hochberg).
4. Run Principal Component Analysis (PCA) with `prcomp()` and interpret variance.



## 1. Wide vs. Long Data Reshaping
`ggplot2` requires tidy 'long' format data for multi-group visualizations.


In [ ]:
# Ensure required libraries
if (!requireNamespace("reshape2", quietly = TRUE)) install.packages("reshape2")
if (!requireNamespace("ggplot2", quietly = TRUE)) install.packages("ggplot2")
library(reshape2)
library(ggplot2)

# Wide expression table
wide_counts <- data.frame(
  Gene = c("CD8A", "CD4", "FOXP3", "MS4A1", "CD14"),
  Tumor_1 = c(140, 210, 85, 420, 310),
  Tumor_2 = c(185, 195, 92, 380, 290),
  Normal_1 = c(25, 45, 12, 110, 85),
  Normal_2 = c(30, 50, 15, 95, 90)
)

print(wide_counts)

# Convert to long format using melt()
long_counts <- melt(wide_counts, id.vars = "Gene", variable.name = "Sample", value.name = "Counts")
long_counts$Group <- ifelse(grepl("^Tumor", long_counts$Sample), "Tumor", "Normal")
head(long_counts, 8)



## 2. Publication-Ready Visualizations with `ggplot2`
`ggplot2` implements the Grammar of Graphics layer by layer.


In [ ]:
# Grouped Boxplot with Jittered Data Points
p1 <- ggplot(long_counts, aes(x = Gene, y = Counts, fill = Group)) +
  geom_boxplot(alpha = 0.7, outlier.shape = NA, position = position_dodge(0.8)) +
  geom_point(position = position_jitterdodge(jitter.width = 0.1, dodge.width = 0.8), size = 2) +
  scale_fill_manual(values = c("Normal" = "#38BDF8", "Tumor" = "#F43F5E")) +
  theme_bw(base_size = 13) +
  labs(
    title = "Immune Marker Expression: Tumor vs Normal",
    subtitle = "Simulated Bulk RNA-seq Validation",
    x = "Gene Marker",
    y = "Normalized Counts"
  ) +
  theme(
    plot.title = element_text(face = "bold", hjust = 0.5),
    plot.subtitle = element_text(hjust = 0.5),
    legend.position = "top"
  )

print(p1)



## 3. Volcano Plot of Differential Expression
Volcano plots plot statistical significance ($-\log_{10} P$) against magnitude of change ($\log_2 \text{Fold Change}$).


In [ ]:
# Simulate a Differential Expression (DEG) dataset of 1,000 genes
set.seed(123)
n_genes <- 1000
deg_data <- data.frame(
  Gene = paste0("Gene_", 1:n_genes),
  log2FC = rnorm(n_genes, mean = 0, sd = 1.8),
  pvalue = runif(n_genes, min = 1e-8, max = 0.8)
)

# Calculate Adjusted P-value (FDR) using Benjamini-Hochberg
deg_data$padj <- p.adjust(deg_data$pvalue, method = "BH")

# Classify significance
deg_data$Significance <- "Not Significant"
deg_data$Significance[deg_data$log2FC > 1.5 & deg_data$padj < 0.05] <- "Up-regulated"
deg_data$Significance[deg_data$log2FC < -1.5 & deg_data$padj < 0.05] <- "Down-regulated"
deg_data$Significance <- factor(deg_data$Significance, levels = c("Up-regulated", "Down-regulated", "Not Significant"))

# Volcano plot
p_volcano <- ggplot(deg_data, aes(x = log2FC, y = -log10(padj), color = Significance)) +
  geom_point(alpha = 0.7, size = 1.8) +
  scale_color_manual(values = c("Up-regulated" = "#EF4444", "Down-regulated" = "#3B82F6", "Not Significant" = "#94A3B8")) +
  geom_vline(xintercept = c(-1.5, 1.5), linetype = "dashed", color = "grey40") +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey40") +
  theme_classic(base_size = 13) +
  labs(
    title = "Differential Expression Volcano Plot",
    x = "Log2 Fold Change",
    y = "-Log10 Adjusted P-value"
  )

print(p_volcano)



## 4. Statistical Hypothesis Testing & PCA
Testing differences between groups (t-test vs Wilcoxon) and exploratory PCA.


In [ ]:
# 1. Parametric (t-test) vs Non-Parametric (Wilcoxon) Test
tumor_exp <- c(45, 52, 60, 48, 55, 70, 65)
normal_exp <- c(15, 18, 12, 22, 19, 14, 20)

# Check Normality
cat("Shapiro-Wilk test (Tumor):", shapiro.test(tumor_exp)$p.value, "
")

# Student t-test
t_res <- t.test(tumor_exp, normal_exp)
cat("Student t-test P-value:", t_res$p.value, "
")

# Wilcoxon Rank-Sum Test
w_res <- wilcox.test(tumor_exp, normal_exp)
cat("Wilcoxon P-value:", w_res$p.value, "
")

# 2. Principal Component Analysis (PCA)
pca_input <- matrix(rnorm(60, mean = rep(c(10, 20), each = 30)), nrow = 6, ncol = 10)
rownames(pca_input) <- paste0("Sample_", 1:6)
colnames(pca_input) <- paste0("Gene_", 1:10)

pca_res <- prcomp(pca_input, scale. = TRUE)
variance_explained <- summary(pca_res)$importance[2, 1:2] * 100

cat("
PCA Variance Explained:
")
cat("PC1:", round(variance_explained[1], 1), "%
")
cat("PC2:", round(variance_explained[2], 1), "%
")

